# 🌿 Plant Disease Detection – Exploratory Data Analysis

**Dataset:** PlantVillage  
**Classes:** 38 (healthy + disease categories across 14 crop species)  
**Total Images:** ~54,000  

---

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

# Add project root to path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

plt.style.use('dark_background')
print('Libraries loaded ✓')

## 1. Dataset Overview

In [ ]:
DATASET_DIR = PROJECT_ROOT / 'data' / 'raw' / 'PlantVillage'

if not DATASET_DIR.exists():
    print(f'⚠️  Dataset not found at {DATASET_DIR}')
    print('Download from: https://www.kaggle.com/datasets/emmarex/plantdisease')
else:
    class_names = sorted([d.name for d in DATASET_DIR.iterdir() if d.is_dir()])
    class_counts = {cls: len(list((DATASET_DIR / cls).glob('*'))) for cls in class_names}
    total = sum(class_counts.values())
    print(f'✓  Classes found : {len(class_names)}')
    print(f'✓  Total images  : {total:,}')
    print(f'✓  Avg per class : {total // len(class_names):,}')

## 2. Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
names  = list(class_counts.keys())
counts = list(class_counts.values())

colors = ['#4CAF50' if 'healthy' in n.lower() else '#EF5350' for n in names]

bars = ax.barh(names, counts, color=colors, edgecolor='none', height=0.7)
ax.set_xlabel('Number of Images', fontsize=12)
ax.set_title('PlantVillage – Class Distribution', fontsize=14, fontweight='bold')
ax.tick_params(axis='y', labelsize=8)

legend_patches = [
    mpatches.Patch(color='#4CAF50', label='Healthy'),
    mpatches.Patch(color='#EF5350', label='Diseased')
]
ax.legend(handles=legend_patches, fontsize=10)

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=7, color='#aaa')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'class_distribution.png', dpi=150)
plt.show()

## 3. Sample Images Grid

In [ ]:
import random
random.seed(42)

selected_classes = random.sample(class_names, min(12, len(class_names)))

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for ax, cls in zip(axes, selected_classes):
    images = list((DATASET_DIR / cls).glob('*.jpg')) + \
             list((DATASET_DIR / cls).glob('*.JPG')) + \
             list((DATASET_DIR / cls).glob('*.png'))
    if images:
        img = Image.open(random.choice(images)).resize((224, 224))
        ax.imshow(img)
    ax.set_title(cls.replace('___', '\n').replace('_', ' '), fontsize=7, color='white')
    ax.axis('off')

for ax in axes[len(selected_classes):]:
    ax.axis('off')

plt.suptitle('Sample Leaf Images from PlantVillage', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Healthy vs. Diseased Split

In [ ]:
healthy_count  = sum(v for k, v in class_counts.items() if 'healthy' in k.lower())
diseased_count = sum(v for k, v in class_counts.items() if 'healthy' not in k.lower())

fig, ax = plt.subplots(figsize=(6, 6))
wedge_colors = ['#4CAF50', '#EF5350']
wedges, texts, autotexts = ax.pie(
    [healthy_count, diseased_count],
    labels=['Healthy', 'Diseased'],
    colors=wedge_colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': '#0d1117', 'linewidth': 3}
)
for text in texts:       text.set_color('white'); text.set_fontsize(13)
for at in autotexts:     at.set_color('white');   at.set_fontsize(12); at.set_fontweight('bold')

ax.set_title('Dataset Composition', fontsize=14, fontweight='bold')
plt.savefig(PROJECT_ROOT / 'notebooks' / 'healthy_vs_diseased.png', dpi=150)
plt.show()

print(f'Healthy  images : {healthy_count:,} ({healthy_count/total*100:.1f}%)')
print(f'Diseased images : {diseased_count:,} ({diseased_count/total*100:.1f}%)')

## 5. Image Dimensions Analysis

In [ ]:
# Sample 200 random images and check their native dimensions
all_images = list(DATASET_DIR.rglob('*.jpg'))[:200]
widths, heights = [], []

for p in all_images:
    try:
        w, h = Image.open(p).size
        widths.append(w)
        heights.append(h)
    except:
        pass

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=20, color='#42A5F5', edgecolor='none')
axes[0].set_title('Image Widths'); axes[0].set_xlabel('Pixels')
axes[1].hist(heights, bins=20, color='#AB47BC', edgecolor='none')
axes[1].set_title('Image Heights'); axes[1].set_xlabel('Pixels')

plt.suptitle('Native Image Dimensions (sample n=200)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Width  – Mean: {np.mean(widths):.0f}px | Range: {min(widths)}–{max(widths)}px')
print(f'Height – Mean: {np.mean(heights):.0f}px | Range: {min(heights)}–{max(heights)}px')

## 6. Crop-Level Disease Breakdown

In [ ]:
crop_disease = {}
for cls, count in class_counts.items():
    crop = cls.split('___')[0].replace('_', ' ')
    crop_disease[crop] = crop_disease.get(crop, 0) + count

sorted_crops = dict(sorted(crop_disease.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(sorted_crops.keys(), sorted_crops.values(),
              color='#66BB6A', edgecolor='none', width=0.6)
ax.set_title('Images per Crop Type', fontsize=13, fontweight='bold')
ax.set_ylabel('Total Images')
plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, sorted_crops.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}', ha='center', fontsize=8, color='#aaa')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'crop_distribution.png', dpi=150)
plt.show()

## 7. Disease Info Database Summary

In [ ]:
from src.utils.disease_info import DISEASE_DATABASE, list_all_diseases

print(f'Total diseases in database : {len(DISEASE_DATABASE)}')
print('\n' + '-'*60)
print(f'{"Label":<45} {"Name"}')
print('-'*60)
for label, info in list(DISEASE_DATABASE.items())[:15]:
    print(f'{label:<45} {info["name"]}')

## 8. Augmentation Preview

In [ ]:
import tensorflow as tf
from src.data.preprocessing import Preprocessor

# Pick a sample image
sample_path = next(DATASET_DIR.rglob('*.jpg'))
orig = Image.open(sample_path).resize((224, 224))

prep = Preprocessor()

# Generate 6 augmented variants
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
axes[0].imshow(orig); axes[0].set_title('Original', color='white'); axes[0].axis('off')

arr = np.array(orig, dtype=np.float32)
aug_layer = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2),
])

for i in range(1, 8):
    aug = aug_layer(tf.expand_dims(arr, 0), training=True)[0].numpy()
    aug = np.clip(aug, 0, 255).astype(np.uint8)
    axes[i].imshow(aug)
    axes[i].set_title(f'Augmented #{i}', color='white', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'notebooks' / 'augmentation_preview.png', dpi=150)
plt.show()

---
## 📊 EDA Summary

| Key Finding | Value |
|---|---|
| Total classes | 38 |
| Estimated total images | ~54,000 |
| Average images/class | ~1,421 |
| Native image resolution | Typically 256×256 px |
| Model input resolution | 224×224 px |
| Class imbalance | Moderate — addressed via augmentation |

> ✅ Dataset is suitable for transfer learning with MobileNetV2.